In [591]:
# Load necessary libraries
import unicodedata
import pandas as pd
import numpy as np
import re

In [592]:
# Read in 1896-2022 data
athlete_bio = pd.read_csv("rawData/Sum_Win_Olympics_1896-2022/Olympic_Athlete_Bio.csv")
athlete_event_results = pd.read_csv("rawData/Sum_Win_Olympics_1896-2022/Olympic_Athlete_Event_Results.csv")
country_codes = pd.read_csv("rawData/Sum_Win_Olympics_1896-2022/Olympics_Country.csv")

In [593]:
print(athlete_event_results.shape[0])
print(athlete_bio.shape[0])

athlete_event_results['medal'].notna().sum()

316834
155861


44687

In [594]:
# Convert country code and country name to a dictionary
noc_country_dict = dict(zip(country_codes['noc'], country_codes['country']))

In [595]:
# Replace current values in the country column with the country values from the dictionary based on country_noc
athlete_event_results['country'] = athlete_event_results['country_noc'].map(noc_country_dict)

In [596]:
# Drop unnecessary columns
athlete_event_results = athlete_event_results.drop(columns=['edition_id',
                                                            'result_id',
                                                            'pos',
                                                            'isTeamSport'
                                                            ])

athlete_bio = athlete_bio.drop(columns=['height',
                                        'weight',
                                        'country',
                                        'country_noc',
                                        'description',
                                        'special_notes'
                                        ])

In [597]:
athlete_event_results.head()

,edition,country_noc,sport,event,athlete,athlete_id,medal,country
0,1908 Summer Olympics,ANZ,Athletics,"100 metres, Men",Ernest Hutcheon,64710,NaN,Australasia
1,1908 Summer Olympics,ANZ,Athletics,"400 metres, Men",Henry Murray,64756,NaN,Australasia
2,1908 Summer Olympics,ANZ,Athletics,"800 metres, Men",Harvey Sutton,64808,NaN,Australasia
3,1908 Summer Olympics,ANZ,Athletics,"800 metres, Men",Guy Haskins,922519,NaN,Australasia
4,1908 Summer Olympics,ANZ,Athletics,"800 metres, Men",Joseph Lynch,64735,NaN,Australasia


In [598]:
athlete_bio.head()

,athlete_id,name,sex,born
0,65649,Ivanka Bonova,Female,4 April 1949
1,112510,Nataliya Uryadova,Female,15 March 1977
2,114973,Essa Ismail Rashed,Male,14 December 1986
3,30359,Péter Boros,Male,12 January 1908
4,50557,Rudolf Piowatý,Male,28 April 1900


In [599]:
# Merge datasets
complete_athlete_info = pd.merge(athlete_event_results, athlete_bio, on='athlete_id', how='left')

# Rename columns
complete_athlete_info = complete_athlete_info.rename(columns={'edition': 'game',
                                                              'sex': 'gender',
                                                              'born': 'birthdate'})

In [600]:
complete_athlete_info.head()

,game,country_noc,sport,event,athlete,athlete_id,medal,country,name,gender,birthdate
0,1908 Summer Olympics,ANZ,Athletics,"100 metres, Men",Ernest Hutcheon,64710,NaN,Australasia,Ernest Hutcheon,Male,17 June 1889
1,1908 Summer Olympics,ANZ,Athletics,"400 metres, Men",Henry Murray,64756,NaN,Australasia,Henry Murray,Male,14 January 1886
2,1908 Summer Olympics,ANZ,Athletics,"800 metres, Men",Harvey Sutton,64808,NaN,Australasia,Harvey Sutton,Male,18 February 1882
3,1908 Summer Olympics,ANZ,Athletics,"800 metres, Men",Guy Haskins,922519,NaN,Australasia,Guy Haskins,Male,23 December 1883
4,1908 Summer Olympics,ANZ,Athletics,"800 metres, Men",Joseph Lynch,64735,NaN,Australasia,Joseph Lynch,Male,22 April 1878


In [601]:
# Extract year column
year_pattern = r'(\d{4})'
complete_athlete_info['year'] = complete_athlete_info['game'].str.extract(year_pattern)

# Extract season column
season_pattern = r'(Summer|Winter|Equestrian|Intercalated)'
complete_athlete_info['season'] = complete_athlete_info['game'].str.extract(season_pattern)

# Drop original game column
complete_athlete_info = complete_athlete_info.drop(columns='game')

In [602]:
# Clean athlete, sport, event, and country columns
def remove_accents(text):
    if pd.isna(text):
        return text
    
    return unicodedata.normalize('NFKD', text).encode('ASCII', 'ignore').decode('utf-8')

complete_athlete_info['athlete'] = complete_athlete_info['athlete'].apply(remove_accents)
complete_athlete_info['sport'] = complete_athlete_info['sport'].apply(remove_accents)
complete_athlete_info['event'] = complete_athlete_info['event'].apply(remove_accents)
complete_athlete_info['country'] = complete_athlete_info['country'].apply(remove_accents)

In [603]:
complete_athlete_info.head(20)

,country_noc,sport,event,athlete,athlete_id,medal,country,name,gender,birthdate,year,season
0,ANZ,Athletics,"100 metres, Men",Ernest Hutcheon,64710,NaN,Australasia,Ernest Hutcheon,Male,17 June 1889,1908,Summer
1,ANZ,Athletics,"400 metres, Men",Henry Murray,64756,NaN,Australasia,Henry Murray,Male,14 January 1886,1908,Summer
2,ANZ,Athletics,"800 metres, Men",Harvey Sutton,64808,NaN,Australasia,Harvey Sutton,Male,18 February 1882,1908,Summer
3,ANZ,Athletics,"800 metres, Men",Guy Haskins,922519,NaN,Australasia,Guy Haskins,Male,23 December 1883,1908,Summer
4,ANZ,Athletics,"800 metres, Men",Joseph Lynch,64735,NaN,Australasia,Joseph Lynch,Male,22 April 1878,1908,Summer
5,ANZ,Athletics,"800 metres, Men",Henry Murray,64756,NaN,Australasia,Henry Murray,Male,14 January 1886,1908,Summer
6,ANZ,Athletics,"1,500 metres, Men",Joseph Lynch,64735,NaN,Australasia,Joseph Lynch,Male,22 April 1878,1908,Summer
7,ANZ,Athletics,"1,500 metres, Men",Charles Swain,79576,NaN,Australasia,Charles Swain,Male,16 January 1885,1908,Summer
8,ANZ,Athletics,"1,500 metres, Men",Guy Haskins,922519,NaN,Australasia,Guy Haskins,Male,23 December 1883,1908,Summer
9,ANZ,Athletics,"1,500 metres, Men",George Blake,64619,NaN,Australasia,George Blake,Male,4 September 1878,1908,Summer


In [604]:
print(complete_athlete_info.shape[0])

complete_athlete_info['medal'].notna().sum()

316834


44687

In [605]:
# Check for duplicates
complete_athlete_info.drop_duplicates()

print(complete_athlete_info.shape[0])

316834


In [606]:
# Drop extra columns
complete_athlete_info = complete_athlete_info.drop(columns=['athlete_id',
                                                            'name'
                                                            ])

# Reorder columns
complete_athlete_info = complete_athlete_info[['year', 
                                               'season', 
                                               'athlete', 
                                               'gender', 
                                               'birthdate',
                                               'country', 
                                               'country_noc', 
                                               'sport',
                                               'event',
                                               'medal']]

# Save to csv
complete_athlete_info.to_csv("complete_athlete_info.csv", index=False)

In [607]:
# Read in 2024 data
athletes_2024 = pd.read_csv('rawData/Paris_2024/Paris_2024_Athletes.csv')
medallists_2024 = pd.read_csv('rawData/Paris_2024/Paris_2024_Medallists.csv')

In [608]:
print(athletes_2024.shape[0])
print(medallists_2024.shape[0])

medallists_2024['medal_type'].notna().sum()

11113
2315


2315

In [609]:
# Merge datasets
joined_2024 = pd.merge(athletes_2024, medallists_2024, left_on='code', right_on='code_athlete', how='left')

print(joined_2024.shape[0])

joined_2024['medal_type'].notna().sum()

11374


2315

In [610]:
# Check for duplicates
joined_2024 = joined_2024.drop_duplicates()

print(joined_2024.shape[0])

11374


In [611]:
joined_2024.tail()

,code,current,name_x,name_short,name_tv,gender_x,function,country_code_x,country_x,country_long_x,...,team,team_gender,discipline,event,event_type,url_event,birth_date_y,code_athlete,code_team,is_medallist
11369,4986655,True,ADA ETO Sefora,ADA ETO S,Sefora ADA ETO,Female,Athlete,GEQ,Equatorial Guinea,Equatorial Guinea,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11370,9460001,True,LIUZZI Emanuela,LIUZZI E,Emanuela LIUZZI,Female,Athlete,ITA,Italy,Italy,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11371,1972077,False,BOERS Isayah,NaN,NaN,Male,Athlete,NED,Netherlands,Netherlands,...,Netherlands,X,Athletics,4 x 400m Relay Mixed,TEAM,/en/paris-2024/results/athletics/4-x-400m-rela...,1999-06-19,1972077.0,ATHX4X400M--NED01,False
11372,1899865,False,STAUT Kevin,NaN,NaN,Male,Athlete,FRA,France,France,...,France,O,Equestrian,Jumping Team,TEAM,/en/paris-2024/results/equestrian/jumping-team...,1980-11-15,1899865.0,EQUOJUMPTEAMFRA01,False
11373,1924402,False,CARVELL Charlie,NaN,NaN,Male,Athlete,GBR,Great Britain,Great Britain,...,Great Britain,M,Athletics,Men's 4 x 400m Relay,TEAM,/en/paris-2024/results/athletics/men-s-4-x-400...,2004-06-30,1924402.0,ATHM4X400M--GBR01,False


In [612]:
# Drop extra columns
joined_2024 = joined_2024.drop(columns=['code',
                                        'current',
                                        'name_short',
                                        'name_tv',
                                        'function',
                                        'country_long_x',
                                        'nationality_x',
                                        'nationality_long_x',
                                        'nationality_code_x',
                                        'height',
                                        'weight',
                                        'birth_place',
                                        'birth_country',
                                        'residence_place',
                                        'residence_country',
                                        'nickname',
                                        'hobbies',
                                        'occupation',
                                        'education',
                                        'family',
                                        'lang',
                                        'coach',
                                        'reason',
                                        'hero',
                                        'influence',
                                        'philosophy',
                                        'sporting_relatives',
                                        'ritual',
                                        'other_sports',
                                        'medal_date',
                                        'medal_code',
                                        'name_y',
                                        'gender_y',
                                        'country_code_y',
                                        'country_y',
                                        'country_long_y',
                                        'nationality_code_y',
                                        'nationality_y',
                                        'nationality_long_y',
                                        'team',
                                        'team_gender',
                                        'discipline',
                                        'event',
                                        'event_type',
                                        'url_event',
                                        'birth_date_y',
                                        'code_athlete',
                                        'code_team',
                                        'is_medallist'])

In [613]:
# Rename columns
joined_2024 = joined_2024.rename(columns={'name_x': 'athlete',
                                          'gender_x': 'gender',
                                          'country_code_x': 'country_noc',
                                          'country_x': 'country',
                                          'disciplines': 'sport',
                                          'events': 'event',
                                          'birth_date_x': 'birthdate',
                                          'medal_type': 'medal'})

In [614]:
joined_2024.tail()

,athlete,gender,country_noc,country,sport,event,birthdate,medal
11369,ADA ETO Sefora,Female,GEQ,Equatorial Guinea,['Athletics'],"[""Women's 100m""]",6/11/2003,NaN
11370,LIUZZI Emanuela,Female,ITA,Italy,['Wrestling'],"[""Women's Freestyle 50kg""]",4/27/2000,NaN
11371,BOERS Isayah,Male,NED,Netherlands,['Athletics'],[4 x 400m Relay Mixed],6/19/1999,Gold Medal
11372,STAUT Kevin,Male,FRA,France,['Equestrian'],[Jumping Team],11/15/1980,Bronze Medal
11373,CARVELL Charlie,Male,GBR,Great Britain,['Athletics'],[Men's 4 x 400m Relay],6/30/2004,Bronze Medal


In [615]:
# Replace current values in the country column with the country values from the dictionary based on country_noc
joined_2024['country'] = joined_2024['country_noc'].map(noc_country_dict)

In [616]:
# Clean athlete, sport, and medal columns
def switch_first_last(name):
    parts = name.strip().split()

    last_name_parts = [part for part in parts if part.isupper()]
    first_name_parts = [part for part in parts if not part.isupper()]

    return ' '.join(first_name_parts + last_name_parts)

def format_athlete_name(name):
    if pd.isna(name):
        return name
    
    def fix_case(match):
        return match.group(0).title()
    
    return re.sub(r'\b[A-ZÁÉÍÓÚÑÜ]{2,}\b', fix_case, name)

joined_2024['athlete'] = joined_2024['athlete'].apply(switch_first_last)
joined_2024['athlete'] = joined_2024['athlete'].apply(format_athlete_name)

joined_2024['medal'] = joined_2024['medal'].str.replace(r'\s*Medal', '', regex=True)

def format_sport_column(sport):
    if pd.isna(sport):
        return sport
    
    items = re.findall(f"'(.*?)'", sport)
    if items:
        return items[-1]
    
    return sport

joined_2024['sport'] = joined_2024['sport'].apply(format_sport_column)

In [617]:
# Add year and season columns
joined_2024['year'] = '2024'
joined_2024['season'] = 'Summer'

In [618]:
joined_2024.head()

,athlete,gender,country_noc,country,sport,event,birthdate,medal,year,season
0,Artur Aleksanyan,Male,ARM,Armenia,Wrestling,"[""Men's Greco-Roman 97kg""]",10/21/1991,Silver,2024,Summer
1,Malkhas Amoyan,Male,ARM,Armenia,Wrestling,"[""Men's Greco-Roman 77kg""]",1/22/1999,Bronze,2024,Summer
2,Slavik Galstyan,Male,ARM,Armenia,Wrestling,"[""Men's Greco-Roman 67kg""]",12/21/1996,NaN,2024,Summer
3,Arsen Harutyunyan,Male,ARM,Armenia,Wrestling,"[""Men's Freestyle 57kg""]",11/22/1999,NaN,2024,Summer
4,Vazgen Tevanyan,Male,ARM,Armenia,Wrestling,"[""Men's Freestyle 65kg""]",10/27/1999,NaN,2024,Summer


In [619]:
print(joined_2024.shape[0])

joined_2024['medal'].notna().sum()

11374


2315

In [620]:
# Reorder columns
joined_2024 = joined_2024[['year', 
                           'season',
                           'athlete',
                           'gender',
                           'birthdate',
                           'country',
                           'country_noc',
                           'sport',
                           'event',
                           'medal']]

# Save to csv
joined_2024.to_csv('joined_2024.csv', index=False)

In [621]:
# Concatenate 1896-2022 dataset with 2024 dataset
final_athlete_game_dataset = pd.concat([complete_athlete_info, joined_2024], ignore_index=True)

In [622]:
# Create age column
final_athlete_game_dataset['birthdate'] = pd.to_datetime(final_athlete_game_dataset['birthdate'], errors='coerce')
final_athlete_game_dataset['year'] = pd.to_numeric(final_athlete_game_dataset['year'], errors='coerce')

final_athlete_game_dataset['age'] = final_athlete_game_dataset['year'] - final_athlete_game_dataset['birthdate'].dt.year

# Drop the original birthdate column
final_athlete_game_dataset = final_athlete_game_dataset.drop(columns=['birthdate'])

In [623]:
print(final_athlete_game_dataset.shape[0])

final_athlete_game_dataset['medal'].notna().sum()

328208


47002

In [624]:
# Check for missing values
final_athlete_game_dataset.isna().sum()

# Reorder columns
final_athlete_game_dataset = final_athlete_game_dataset[['year', 
                                                         'season',
                                                         'athlete',
                                                         'gender',
                                                         'age',
                                                         'country',
                                                         'country_noc',
                                                         'sport',
                                                         'event',
                                                         'medal']]

In [625]:
# Save to csv
final_athlete_game_dataset.to_csv("final_athlete_game_dataset.csv", index=False)